<a href="https://colab.research.google.com/github/bongwony/ku_analyzer_kiwi/blob/main/%ED%95%9C%EA%B5%AD%EC%96%B4%EB%B0%9C%ED%99%94%EB%B6%84%EC%84%9D%EA%B8%B0_kiwi_v12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 한국어 발화 분석기 v12 (Kiwi 버전)
이 도구는 언어치료 및 발화 분석 연구를 위해 제작되었습니다.
- **분석 엔진:** Kiwi (Korean Intelligent Word Analyzer)
- **산출 지표:** 발화수 / Token / Type / MLU-e / MLU-w / MLU-m / TTR_전체 / TTR_내용어 / NDW / NDW-50 / 내용어수 / 기능어수
- **기능:** 발화별 형태소 분석 및 Excel 내보내기 (발화별 상세 + 요약 통계 시트)
- **옵션:** 파생접사(XSV·XSA·XSN·XSM·XPN) 분석 ON/OFF

| 지표 | 설명 | 기준 |
|------|------|------|
| 발화수 | 총 발화 수 | — |
| Token | 총 형태소 수 | 표면형, 문장부호 제외 |
| Type | 형태소 유형 수 | 표면형 |
| **MLU-e** | 평균발화길이 (어절) | 어절/발화 (공백 분리) |
| **MLU-w** | 평균발화길이 (단어) | 단어/발화 (학교문법: 조사를 단어로 인정) |
| **MLU-m** | 평균발화길이 (형태소) | 형태소/발화 |
| TTR_전체 | 유형-토큰 비율 | Type/Token |
| TTR_내용어 | 내용어 어휘 다양도 | 내용어 Type/Token |
| NDW | 다른 단어 수 | 내용어 표면형 기준 |
| NDW-50 | 첫 50 내용어 기준 NDW | 표본 크기 보정 |
| 내용어수 | 명사·동사·형용사·부사 등 | NNG NNP NNB NP NR VV VA MM MAG MAJ |
| 기능어수 | 조사·어미·보조용언 등 | 나머지 형태소 |

### MLU 세 가지 산정 방식 비교 (예: '시간이 있다')
| 방식 | 분석 결과 | 길이 |
|------|-----------|------|
| MLU-e (어절) | `시간이` / `있다` | **2** |
| MLU-w (단어, 학교문법) | `시간` / `이` / `있다` | **3** |
| MLU-m (형태소) | `시간` / `이` / `있` / `다` | **4** |

### 접사 분석 옵션 (예: '공부하다')
| 옵션 | 형태소 분석 | 형태소 수 |
|------|-------------|----------|
| **ON** (분석함, 기본값) | `공부/NNG` + `하/XSV` + `다/EF` | 3 |
| **OFF** (결합) | `공부하/VV` + `다/EF` | 2 |

### 발화 분리 규칙
- **줄바꿈은 강제 발화 경계**: 사용자가 엔터로 구분한 줄은 항상 별개 발화로 처리됩니다.
- **한 줄 안에 여러 발화**가 있으면 Kiwi가 구두점·문법 단서로 추가 분리합니다.
- 빈 줄과 앞뒤 공백은 무시됩니다.
- 따라서 입력 시 엔터 구분이 필수는 아니지만, 구두점 없는 구어체 발화(예: 아동발화)는 엔터로 구분해야 정확히 분석됩니다.

In [1]:
# 1. 필수 라이브러리 설치
!pip install kiwipiepy pandas ipywidgets -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 8.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 135.1 MB/s eta 0:00:00


In [2]:
import pandas as pd
import datetime
from kiwipiepy import Kiwi
from ipywidgets import widgets, Layout
from IPython.display import display, clear_output, HTML
from google.colab import files

kiwi = Kiwi()

# ──────────────────────────────────────────────────────
# 태그 집합 정의
# ──────────────────────────────────────────────────────

# 문장부호·기호 (분석에서 완전 제외)
PUNCT_TAGS = {'SF', 'SP', 'SS', 'SE', 'SO', 'SW', 'SB', 'UNKNOWN'}

# 내용어 태그 (Kiwi 기준)
#   명사류: NNG NNP NNB NP NR
#   용언류: VV VA  (VX 보조용언은 기능어로 분류)
#   수식언: MM MAG MAJ
CONTENT_TAGS = {'NNG', 'NNP', 'NNB', 'NP', 'NR',
                'VV',  'VA',
                'MM',  'MAG', 'MAJ'}

# ──────────────────────────────────────────────────────
# 학교문법 기반 '단어' 단위 태그 정의 (MLU-w 산정용)
# ──────────────────────────────────────────────────────
WORD_TAGS = {
    # 체언
    'NNG', 'NNP', 'NNB', 'NP', 'NR',
    # 용언 (어간 = 단어 1)
    'VV', 'VA', 'VX', 'VCP', 'VCN',
    # 수식언
    'MM', 'MAG', 'MAJ',
    # 독립언
    'IC',
    # 조사 (학교문법에서 단어로 인정)
    'JKS', 'JKC', 'JKG', 'JKO', 'JKB', 'JKV', 'JKQ', 'JX', 'JC',
    # 자립형식
    'SL', 'SH', 'SN',
}


# ──────────────────────────────────────────────────────
# 태그 정규화 (Kiwi는 불규칙 활용을 'VV-I', 'XSA-I' 등으로 태그함)
# ──────────────────────────────────────────────────────
def base_tag(tag):
    """'VV-I' → 'VV', 'XSA-I' → 'XSA' 처럼 접미사를 떼어낸 기본 태그를 반환."""
    return tag.split('-')[0]


# ──────────────────────────────────────────────────────
# 파생접사 결합 (옵션 OFF용)
# ──────────────────────────────────────────────────────
# 파생접미사: 앞 어근에 결합. 결합 후 태그는 접미사 종류에 따라 결정
DERIV_SUFFIX_TAGS = {'XSV', 'XSA', 'XSN', 'XSM'}
SUFFIX_TO_TAG = {
    'XSV': 'VV',   # 동사파생  (예: 공부+하 → 공부하/VV)
    'XSA': 'VA',   # 형용사파생 (예: 행복+하 → 행복하/VA)
    'XSN': 'NNG',  # 명사파생  (예: 가능+성 → 가능성/NNG)
    'XSM': 'MAG',  # 부사파생
}
# 파생접두사: 뒤 어근에 결합. 결합 후 태그는 어근 태그를 유지
DERIV_PREFIX_TAGS = {'XPN'}


class _Tok:
    """결합 후 토큰을 표현하기 위한 가벼운 클래스."""
    __slots__ = ('form', 'tag')
    def __init__(self, form, tag):
        self.form = form
        self.tag  = tag


def merge_derivational_affixes(tokens):
    """파생접사를 인접 어근에 결합한 토큰 리스트를 반환.

    - XSV/XSA/XSN/XSM: 앞 토큰과 결합, 태그는 SUFFIX_TO_TAG 매핑
    - XPN: 뒤 토큰과 결합, 태그는 뒤 토큰 태그 유지
    - 어근 없이 단독으로 등장하는 접사는 그대로 유지
    """
    out = []
    toks = list(tokens)
    i = 0
    n = len(toks)
    while i < n:
        cur = toks[i]
        cur_base = base_tag(cur.tag)

        # 접두사 + 다음 어근 결합
        if cur_base in DERIV_PREFIX_TAGS and i + 1 < n:
            nxt = toks[i + 1]
            out.append(_Tok(cur.form + nxt.form, nxt.tag))
            i += 2
            continue

        # 일반 토큰 + 다음 토큰이 파생접미사면 결합
        if i + 1 < n:
            nxt_base = base_tag(toks[i + 1].tag)
            if nxt_base in DERIV_SUFFIX_TAGS:
                new_tag = SUFFIX_TO_TAG.get(nxt_base, cur.tag)
                out.append(_Tok(cur.form + toks[i + 1].form, new_tag))
                i += 2
                continue

        out.append(_Tok(cur.form, cur.tag))
        i += 1
    return out


# ──────────────────────────────────────────────────────
# 핵심 분석 함수
# ──────────────────────────────────────────────────────
# ──────────────────────────────────────────────────────
# 발화 분리 (줄바꿈 강제 경계 + Kiwi 자동 분리)
# ──────────────────────────────────────────────────────
def split_utterances(text):
    """입력 텍스트를 발화 단위로 분리.

    - 줄바꿈은 강제 발화 경계로 존중 (사용자가 명시한 경계는 절대 합쳐지지 않음).
    - 한 줄 안에 여러 발화가 있으면 Kiwi의 split_into_sents()로 추가 분리.
    - 빈 줄과 앞뒤 공백은 무시.

    이렇게 하면 구두점이 없는 구어체 발화(예: '갈래', '가자', '안 돼')도
    사용자가 엔터로 구분하기만 하면 정확히 발화별로 분석됩니다.
    """
    utterances = []
    for line in text.split('\n'):
        line = line.strip()
        if not line:
            continue
        sents = kiwi.split_into_sents(line)
        for s in sents:
            t = s.text.strip()
            if t:
                utterances.append(t)
    return utterances


def run_analysis(text_content, analyze_affixes=True):
    """발화 분석 메인 함수.

    Args:
        text_content (str): 분석할 텍스트
        analyze_affixes (bool): True면 파생접사를 별개 형태소로 분석 (기본값).
                                False면 접사를 어근에 결합해 단일 토큰으로 처리.
    """
    utterances = split_utterances(text_content)
    if not utterances:
        return None, None

    results       = []
    all_tokens_sf = []   # 전체 형태소 표면형  (문장부호 제외)
    all_cont_forms = []  # 내용어 형태소 표면형
    all_func_sf    = []  # 기능어 표면형

    for i, utt in enumerate(utterances):
        tokens = kiwi.tokenize(utt)

        # 옵션 적용: 접사 분석 OFF면 파생접사를 어근에 결합
        if not analyze_affixes:
            tokens = merge_derivational_affixes(tokens)

        # 문장부호 제외 (= 형태소 토큰)
        morph_tokens = [t for t in tokens if base_tag(t.tag) not in PUNCT_TAGS]

        # 내용어 / 기능어 분류 (형태소 기준)
        cont_tok = [t for t in morph_tokens if base_tag(t.tag) in CONTENT_TAGS]
        func_tok = [t for t in morph_tokens if base_tag(t.tag) not in CONTENT_TAGS]

        # 학교문법 '단어' 토큰: 어미·접사·문장부호 제외, 조사는 포함
        word_tokens = [t for t in morph_tokens if base_tag(t.tag) in WORD_TAGS]

        # 표면형
        sf_all     = [t.form for t in morph_tokens]
        cont_forms = [t.form for t in cont_tok]
        sf_func    = [t.form for t in func_tok]

        all_tokens_sf.extend(sf_all)
        all_cont_forms.extend(cont_forms)
        all_func_sf.extend(sf_func)

        morph_str = " ".join(f"{t.form}/{t.tag}" for t in morph_tokens)

        results.append({
            "No":          i + 1,
            "발화":        utt,
            "어절수":      len(utt.split()),       # MLU-e 단위
            "단어수":      len(word_tokens),       # MLU-w 단위 (학교문법)
            "형태소수":    len(morph_tokens),      # MLU-m 단위
            "내용어수":    len(cont_tok),
            "기능어수":    len(func_tok),
            "형태소분석":  morph_str,
            "_cont_forms": cont_forms,
        })

    df    = pd.DataFrame(results)
    n_utt = len(df)

    # ── 전체 요약 지표 ──────────────────────────────────
    token_n   = len(all_tokens_sf)
    type_n    = len(set(all_tokens_sf))
    cont_n    = len(all_cont_forms)
    func_n    = token_n - cont_n
    ndw       = len(set(all_cont_forms))
    ndw_50    = len(set(all_cont_forms[:50]))

    summary = {
        '발화수':       n_utt,
        'Token':        token_n,
        'Type':         type_n,
        'MLU_e':        round(df['어절수'].mean(),   2),
        'MLU_w':        round(df['단어수'].mean(),   2),
        'MLU_m':        round(df['형태소수'].mean(), 2),
        'TTR_전체':     round(type_n / token_n, 4)               if token_n else 0,
        'TTR_내용어':   round(len(set(all_cont_forms)) / cont_n, 4) if cont_n else 0,
        'NDW':          ndw,
        'NDW_50':       ndw_50,
        '내용어수':      cont_n,
        '기능어수':      func_n,
        '_접사분석':     analyze_affixes,  # 메타 정보 (다운로드 라벨용)
    }
    return df, summary


# ──────────────────────────────────────────────────────
# UI 위젯
# ──────────────────────────────────────────────────────
output_view         = widgets.Output()
analyzed_dataframes = {}

# 접사 분석 옵션 체크박스 (양쪽 탭에서 공유)
affix_checkbox = widgets.Checkbox(
    value=True,
    description="파생접사 분석 (예: 공부/하/다 vs 공부하/다)",
    indent=False,
    layout=Layout(width="98%", margin="0 0 8px 0")
)
affix_help = widgets.HTML(
    "<div style='font-size:11px;color:#64748b;margin-bottom:8px;padding-left:24px;'>"
    "<b>체크 ON</b>: 파생접사(XSV·XSA·XSN·XSM·XPN)를 별개 형태소로 분석 (기본값) &nbsp;|&nbsp; "
    "<b>체크 OFF</b>: 접사를 어근에 결합 (예: 공부+하/XSV → 공부하/VV)"
    "</div>"
)

text_input       = widgets.Textarea(
    placeholder="발화를 입력하세요. 구두점이 없는 구어체는 엔터로 구분해 주세요.",
    layout=Layout(width="98%", height="200px")
)
analyze_text_btn = widgets.Button(description="📝 입력 텍스트 분석", button_style="primary")
file_upload      = widgets.FileUpload(accept=".txt", multiple=False)
analyze_file_btn = widgets.Button(description="📁 업로드 파일 분석", button_style="info")

tabs = widgets.Tab(children=[
    widgets.VBox([text_input, analyze_text_btn]),
    widgets.VBox([widgets.HTML("<b>.txt 파일을 선택해 주세요:</b>"), file_upload, analyze_file_btn])
])
tabs.set_title(0, "직접 입력")
tabs.set_title(1, "파일 업로드")

clear_results_btn = widgets.Button(
    description="🗑️ 전체 결과 지우기",
    button_style="danger",
    layout=Layout(width="auto", margin="10px 0")
)


# ──────────────────────────────────────────────────────
# 결과 출력 함수
# ──────────────────────────────────────────────────────
def _stat_box(label, value, note="", color="#1e293b"):
    return (
        f"<div style='text-align:center;min-width:80px;padding:4px 8px;'>"
        f"<b style='font-size:12px;color:#64748b;'>{label}</b><br>"
        f"<span style='font-size:22px;color:{color};font-weight:700;'>{value}</span>"
        + (f"<br><span style='font-size:10px;color:#94a3b8;'>{note}</span>" if note else "")
        + "</div>"
    )

def display_results(df, summary, source_title="분석 결과"):
    timestamp  = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    safe_title = source_title.replace(" ","_").replace("/","_").replace(":","").replace("'","")
    unique_key = f"speech_analysis_{safe_title}_{timestamp}"
    analyzed_dataframes[unique_key] = df.copy()

    affix_on = summary.get('_접사분석', True)
    affix_label = "접사분석 ON" if affix_on else "접사결합 (접사분석 OFF)"
    affix_color = "#0891b2" if affix_on else "#d97706"

    with output_view:
        display(HTML(
            f"<div style='margin-top:20px;padding:10px;background:#e0f2f7;"
            f"border-left:5px solid #2196f3;font-weight:bold;font-size:1.1em;'>"
            f"{source_title} "
            f"<span style='background:{affix_color};color:white;padding:2px 8px;"
            f"border-radius:4px;font-size:0.78em;margin-left:8px;'>{affix_label}</span>"
            f"</div>"
        ))

        # ─ 요약 통계 ────────────────────────────────────
        s = summary
        row1 = "".join([
            _stat_box("발화수",    s["발화수"],  "utterances"),
            _stat_box("Token",     s["Token"],   "총 형태소 수", "#2563eb"),
            _stat_box("Type",      s["Type"],    "형태소 유형 수", "#2563eb"),
            _stat_box("MLU-e",     s["MLU_e"],   "어절/발화",     "#7c3aed"),
            _stat_box("MLU-w",     s["MLU_w"],   "단어/발화",     "#7c3aed"),
            _stat_box("MLU-m",     s["MLU_m"],   "형태소/발화",   "#7c3aed"),
        ])
        row2 = "".join([
            _stat_box("TTR 전체",  s["TTR_전체"],  "Type/Token",       "#0891b2"),
            _stat_box("TTR 내용어",s["TTR_내용어"],"내용어 Type/Token", "#0891b2"),
            _stat_box("NDW",       s["NDW"],       "다른 단어 수", "#059669"),
            _stat_box("NDW-50",    s["NDW_50"],    "첫 50 내용어",    "#059669"),
            _stat_box("내용어수",  s["내용어수"],  "content words"),
            _stat_box("기능어수",  s["기능어수"],  "function words"),
        ])

        summary_html = f"""
        <div style="margin:16px 0;border:1px solid #cbd5e1;border-radius:10px;
                    overflow:hidden;font-family:sans-serif;">
          <div style="background:#1e3a5f;color:white;padding:10px 15px;font-weight:bold;">
            📊 전체 요약 통계
            <span style="font-weight:normal;font-size:0.82em;opacity:0.8;">
              (MLU-w는 학교문법 기준 '단어'(조사 별개) 단위)
            </span>
          </div>
          <div style="background:#f8fafc;padding:12px 8px;border-bottom:1px solid #e2e8f0;">
            <div style="font-size:11px;color:#94a3b8;margin-bottom:6px;padding-left:6px;">
              ▸ 발화 / 형태소 기반
            </div>
            <div style="display:flex;flex-wrap:wrap;justify-content:flex-start;">{row1}</div>
          </div>
          <div style="background:white;padding:12px 8px;">
            <div style="font-size:11px;color:#94a3b8;margin-bottom:6px;padding-left:6px;">
              ▸ 어휘 다양도 / 내용어·기능어
            </div>
            <div style="display:flex;flex-wrap:wrap;justify-content:flex-start;">{row2}</div>
          </div>
        </div>
        """
        display(HTML(summary_html))

        # ─ 발화별 상세 테이블 ───────────────────────────
        print("📋 발화별 상세 지표")
        display_cols = ["No","발화","어절수","단어수","형태소수","내용어수","기능어수","형태소분석"]
        display(df[display_cols])

        # ─ Excel 다운로드 ─────────────────────────────────
        dl_btn = widgets.Button(
            description=f"⬇️ '{source_title}' Excel 저장",
            button_style="success",
            layout=Layout(width="auto", margin="10px 0")
        )

        def on_download(b, key=unique_key, title=source_title, summ=summary):
            data = analyzed_dataframes.get(key)
            if data is not None:
                src_label = '직접입력' if '직접' in title else title.split("'")[1].rsplit('.', 1)[0] if "'" in title else 'unknown'
                affix_tag = '접사ON' if summ.get('_접사분석', True) else '접사OFF'
                date_str = datetime.datetime.now().strftime('%Y%m%d')
                fname = f"발화분석_{src_label}_{affix_tag}_{date_str}.xlsx"
                detail_df = data.drop(columns=["_cont_forms"], errors="ignore")

                summary_df = pd.DataFrame([
                    {"지표": "발화수",     "값": summ["발화수"],    "설명": "총 발화 수"},
                    {"지표": "Token",      "값": summ["Token"],     "설명": "총 형태소 수 (문장부호 제외)"},
                    {"지표": "Type",       "값": summ["Type"],      "설명": "형태소 유형 수"},
                    {"지표": "MLU-e",      "값": summ["MLU_e"],     "설명": "평균발화길이 (어절/발화)"},
                    {"지표": "MLU-w",      "값": summ["MLU_w"],     "설명": "평균발화길이 (단어/발화, 학교문법: 조사 별개 단어)"},
                    {"지표": "MLU-m",      "값": summ["MLU_m"],     "설명": "평균발화길이 (형태소/발화)"},
                    {"지표": "TTR_전체",   "값": summ["TTR_전체"],  "설명": "Type / Token"},
                    {"지표": "TTR_내용어", "값": summ["TTR_내용어"],"설명": "내용어 Type/Token"},
                    {"지표": "NDW",        "값": summ["NDW"],       "설명": "다른 단어 수 (내용어)"},
                    {"지표": "NDW-50",     "값": summ["NDW_50"],    "설명": "첫 50 내용어 기준 NDW"},
                    {"지표": "내용어수",   "값": summ["내용어수"],  "설명": "내용어 토큰 수"},
                    {"지표": "기능어수",   "값": summ["기능어수"],  "설명": "기능어 토큰 수"},
                    {"지표": "접사 분석",  "값": "ON" if summ.get('_접사분석', True) else "OFF",
                                          "설명": "파생접사(XSV·XSA·XSN·XSM·XPN)를 별개 형태소로 분석할지 여부"},
                ])

                with pd.ExcelWriter(fname, engine="openpyxl") as writer:
                    detail_df.to_excel(writer, sheet_name="발화별 상세", index=False)
                    summary_df.to_excel(writer, sheet_name="요약 통계", index=False)
                files.download(fname)

        dl_btn.on_click(on_download)
        display(dl_btn)


# ──────────────────────────────────────────────────────
# 이벤트 핸들러
# ──────────────────────────────────────────────────────
def on_text_analyze(b):
    df, summary = run_analysis(text_input.value, analyze_affixes=affix_checkbox.value)
    if df is not None:
        display_results(df, summary, source_title="직접 입력 텍스트 분석 결과")

def on_file_analyze(b):
    if not file_upload.value:
        return
    fname = list(file_upload.value.keys())[0]
    text  = list(file_upload.value.values())[0]["content"].decode("utf-8")
    df, summary = run_analysis(text, analyze_affixes=affix_checkbox.value)
    if df is not None:
        display_results(df, summary, source_title=f"파일 '{fname}' 분석 결과")

def on_clear_results(b):
    with output_view:
        clear_output()
    analyzed_dataframes.clear()
    text_input.value = ""

analyze_text_btn.on_click(on_text_analyze)
analyze_file_btn.on_click(on_file_analyze)
clear_results_btn.on_click(on_clear_results)

# ──────────────────────────────────────────────────────
# 실행
# ──────────────────────────────────────────────────────
print("🗣️ 한국어 발화 분석기 v12 (Kiwi 버전)")
print("   MLU 산출 방식: MLU-e(어절) / MLU-w(단어, 학교문법) / MLU-m(형태소)")
print("   옵션: 파생접사 분석 ON/OFF")
display(affix_checkbox)
display(affix_help)
display(tabs)
display(clear_results_btn)
display(output_view)


🗣️ 한국어 발화 분석기 v12 (Kiwi 버전)
   MLU 산출 방식: MLU-e(어절) / MLU-w(단어, 학교문법) / MLU-m(형태소)
   옵션: 파생접사 분석 ON/OFF


Checkbox(value=True, description='파생접사 분석 (예: 공부/하/다 vs 공부하/다)', indent=False, layout=Layout(margin='0 0 8px 0…

HTML(value="<div style='font-size:11px;color:#64748b;margin-bottom:8px;padding-left:24px;'><b>체크 ON</b>: 파생접사(…

Button(button_style='danger', description='🗑️ 전체 결과 지우기', layout=Layout(margin='10px 0', width='auto'), style=…

Output()